<a href="https://colab.research.google.com/github/alaaguedda/Khadra/blob/main/medical_summerizer_trained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kaggle
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d aminexdr/bhc-mimic-iv-summary
!unzip bhc-mimic-iv-summary.zip


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/aminexdr/bhc-mimic-iv-summary
License(s): unknown
 89% 397M/446M [00:05<00:00, 79.0MB/s]
100% 446M/446M [00:05<00:00, 86.8MB/s]
Archive:  bhc-mimic-iv-summary.zip
  inflating: BHC_MIMIC-IV.csv        


In [ ]:
import pandas as pd

df = pd.read_csv("BHC_MIMIC-IV.csv")


In [ ]:
df = df[['input', 'target']].dropna()

# Shuffle first
df = df.sample(frac=1, random_state=42)

# Take only 10,000 samples
df = df.iloc[:10000]

df.shape

(10000, 2)

In [ ]:
df.head(
)

,input,target
267505,write a discharge summary:\nHistory of Present...,Patient arrived on the unit intubated and seda...
180880,generate a brief hospital summary:\nChief Comp...,"AP: year old female with ho CLL, PAF, not on ..."
252848,create a summary based on the following inform...,Mrs. is a yr old female presenting with new ...
112160,write a discharge summary:\nChief Complaint: C...,Patient had recurring chest pain consistent wi...
99808,create a summary based on the following inform...,"Mr. is a M w ho CAD , atrial fibrillation sp..."


In [ ]:
df.isnull().sum()
df = df.dropna()

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

In [ ]:
!pip install transformers datasets evaluate rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=9b3ad769158d87625ddb2f08c58d78e185c939c46021caeb11125f0ba270d579
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

model_name = "t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
max_input_length = 256
max_target_length = 64

def preprocess_function(examples):
    inputs = ["summarize: " + doc for doc in examples["input"]]

    model_inputs = tokenizer(
        inputs,
        max_length=256,
        truncation=True
    )

    labels = tokenizer(
        examples["target"],
        max_length=64,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=2,
    eval_accumulation_steps=10,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    weight_decay=0.01,
    fp16=True,
    predict_with_generate=True,  # Now this will work!
    generation_max_length=64,
    generation_num_beams=2,
    logging_steps=200,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,6.474880,3.011395


TrainOutput(global_step=500, training_loss=6.467716552734375, metrics={'train_runtime': 114.1317, 'train_samples_per_second': 70.094, 'train_steps_per_second': 4.381, 'total_flos': 541367205888000.0, 'train_loss': 6.467716552734375, 'epoch': 1.0})

In [ ]:
!pip install evaluate rouge_score absl-py

In [ ]:
!pip install bert-score nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.5 MB/s eta 0:00:00


In [ ]:
import torch
import gc

# 1. Clear Python garbage collector
gc.collect()

# 2. Clear NVIDIA cache
torch.cuda.empty_cache()



In [ ]:
import evaluate
import numpy as np

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
bertscore = evaluate.load("bertscore")
meteor = evaluate.load("meteor")

def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    rouge_result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    bleu_result = bleu.compute(
        predictions=[pred.split() for pred in decoded_preds],
        references=[[label.split()] for label in decoded_labels]
    )

    meteor_result = meteor.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    bert_result = bertscore.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        lang="en"
    )

    return {
        "ROUGE-1": rouge_result["rouge1"],
        "ROUGE-2": rouge_result["rouge2"],
        "ROUGE-L": rouge_result["rougeL"],
        "BLEU": bleu_result["bleu"],
        "METEOR": meteor_result["meteor"],
        "BERTScore-F1": np.mean(bert_result["f1"])
    }

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
results = trainer.evaluate()

In [ ]:
import torch

def generate_summary(text):
    device = model.device  # automatically detect cuda or cpu

    input_text = "summarize: " + text

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=256,   # match your training max length
        truncation=True
    )

    # Move inputs to same device as model
    inputs = {key: value.to(device) for key, value in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_length=64,   # match your training target length
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
generate_summary(df['input'].iloc[0])